In [28]:
import argparse
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from model.dataset import BagsDataset, custom_collate_fn
from model.model import MIL, EarlyStopping
import scanpy as sc
import numpy as np

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [36]:
input_dir = '/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/VisiumHD/HumanLungCancer'
adata = sc.read_h5ad(os.path.join(input_dir, 'T_cell.h5ad'))
output_dir = os.path.join('pretrained_models', input_dir.split('/')[-1])
os.makedirs(output_dir, exist_ok=True)

In [37]:
dataset = BagsDataset(adata, immune_cell='tcell', radius=150, n_genes=20000, resolution='high')


['0', '1', '2']
Categories (3, object): ['0', '1', '2']
Tumor cells shape after filtering: (71423, 18085)
Selecting top 20000 genes based on mean expression
Top 3617 genes: Index(['MT-ND4', 'MT-CO2', 'CARTPT', 'MT-CO3', 'MT-ND4L', 'MT-ATP6', 'MT-CYB',
       'CHGA', 'SCG2', 'MT-ND2',
       ...
       'TIMM9', 'EAPP', 'FARP2', 'DNAJC14', 'PDE9A', 'RSL1D1', 'GAS2', 'TTL',
       'SS18', 'DCAKD'],
      dtype='object', length=3617)
Preprocessed data: (150497, 3617)


Creating Bags with radius 150: 100%|██████████████████████| 150497/150497 [01:34<00:00, 1592.56it/s]

Total bags created: 72314
Average instances per bag: 162


In [38]:
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


In [39]:
#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)
from torch.utils.data import Subset
import random

def balance_dataset(dataset, label_ratio=5):
    # Separate bags by labels
    label_0_indices = []
    label_1_indices = []

    for idx in range(len(dataset)):
        _, _, label, _, _, _ = dataset[idx]
        if label.item() == 0:
            label_0_indices.append(idx)
        else:
            label_1_indices.append(idx)

    # Calculate the number of label=0 bags to keep
    num_label_1 = len(label_1_indices)
    num_label_0_to_keep = min(len(label_0_indices), label_ratio * num_label_1)

    # Randomly sample label=0 indices
    sampled_label_0_indices = random.sample(label_0_indices, num_label_0_to_keep)

    # Combine sampled label=0 indices and all label=1 indices
    balanced_indices = sampled_label_0_indices + label_1_indices

    # Shuffle the combined indices
    random.shuffle(balanced_indices)

    # Return a Subset of the original dataset
    return Subset(dataset, balanced_indices)

# Balance train and validation datasets
balanced_train_dataset = balance_dataset(train_dataset, label_ratio=10)
balanced_val_dataset = balance_dataset(val_dataset, label_ratio=10)

# Create new DataLoaders with balanced datasets
train_loader = DataLoader(balanced_train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(balanced_val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

# Print the new label distribution
train_labels_count = sum(1 for _, _, label, _, _, _ in train_loader if label.item() == 1)
train_label_0_count = sum(1 for _, _, label, _, _, _ in train_loader if label.item() == 0)
val_labels_count = sum(1 for _, _, label, _, _, _ in val_loader if label.item() == 1)
val_label_0_count = sum(1 for _, _, label, _, _, _ in val_loader if label.item() == 0)

print(f"Train dataset - Negative: {train_label_0_count}, Positive: {train_labels_count}")
print(f"Validation dataset - Negative: {val_label_0_count}, Postive: {val_labels_count}")


Train dataset - Negative: 6690, Positive: 669
Validation dataset - Negative: 2910, Postive: 291


In [40]:
distances, gene_expressions, labels, core_index, gene_names, cell_ids = next(iter(train_loader))


In [41]:
labels

tensor([0.])

In [42]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=0)
        self.softmax = nn.Softmax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = self.a
        x = self.softmax(-torch.exp(a) * x)
        return x

In [43]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[0.0000e+00],
        [1.1231e-24],
        [0.0000e+00],
        [0.0000e+00],
        [2.5000e-01],
        [0.0000e+00],
        [3.5943e-42],
        [3.5943e-42],
        [0.0000e+00],
        [0.0000e+00],
        [3.2313e-20],
        [1.1231e-24],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [3.5943e-42],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [3.5943e-42],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [7.2710e-36],
        [0.0000e+00],
        [3.2313e-20],
        [4.1766e-39],
        [3.7501e-09],
        [0.0000e+00],
        [0.0000e+00],
        [0

In [44]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x

In [45]:
gene_expressions[0]

tensor([[5.0475, 4.5814, 4.4399,  ..., 0.0000, 0.0000, 0.0000],
        [4.7864, 4.9394, 5.2429,  ..., 0.0000, 0.0000, 0.0000],
        [5.0198, 4.7343, 4.8388,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [4.3857, 4.3293, 5.2945,  ..., 0.0000, 0.0000, 0.0000],
        [4.7841, 4.8966, 5.2258,  ..., 0.0000, 0.0000, 0.0000],
        [4.6913, 4.6405, 4.2654,  ..., 0.0000, 0.0000, 0.0000]])

In [46]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output)

tensor([[1.5476e-01, 4.3587e-02, 2.9667e-02,  ..., 1.7017e-07, 1.7017e-07,
         1.7017e-07],
        [9.8764e-02, 1.4968e-01, 3.4155e-01,  ..., 2.2084e-07, 2.2084e-07,
         2.2084e-07],
        [2.7936e-01, 1.2857e-01, 1.7080e-01,  ..., 3.3125e-07, 3.3125e-07,
         3.3125e-07],
        ...,
        [4.0774e-02, 3.4976e-02, 4.8222e-01,  ..., 2.7096e-07, 2.7096e-07,
         2.7096e-07],
        [9.4445e-02, 1.2821e-01, 3.1376e-01,  ..., 2.1250e-07, 2.1250e-07,
         2.1250e-07],
        [1.9102e-01, 1.6638e-01, 6.0026e-02,  ..., 5.5318e-07, 5.5318e-07,
         5.5318e-07]], grad_fn=<SoftmaxBackward0>)


In [47]:
class Immunogenicity(nn.Module):
    def __init__(self, all_genes):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
        self.ig = nn.Parameter(torch.full((len(all_genes),), -1.0), requires_grad=True)
    
    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        ig = torch.sigmoid(self.ig[indices])
        return ig, filtered_genes

In [48]:
all_genes = pd.read_csv('data/human_filtered.csv')
all_genes = all_genes['Gene'].values.tolist()

In [49]:
model = Immunogenicity(all_genes)

In [50]:
output, filtered_genes = model(list(gene_names[0]))
print(len(output))
print(filtered_genes)
df = pd.DataFrame({'Gene': filtered_genes})


2406
['CARTPT', 'CHGA', 'SCG2', 'PTPRN', 'CHGB', 'SYT13', 'CPLX2', 'SCG5', 'STAT1', 'LENG8', 'CELF3', 'MS4A8', 'NPTN', 'EPCAM', 'IFI6', 'RNASEK', 'KIAA1324', 'SLC22A17', 'GRAMD4', 'NTN4', 'VGF', 'FST', 'LXN', 'CACNA2D3', 'GGA2', 'SMOC1', 'GNG4', 'FAT1', 'FOXA1', 'ATP2C1', 'APLP1', 'PRKACA', 'STAU2', 'PFN2', 'GABRB3', 'CRYBA2', 'SEZ6L2', 'OBSL1', 'H1FX', 'GNG10', 'TUSC3', 'RABGGTB', 'CCNL2', 'H3F3A', 'DCAF16', 'IRX3', 'SREBF2', 'ATP6AP1', 'ADAR', 'EIF4A1', 'LSS', 'ANG', 'ARRB1', 'H1F0', 'VEGFB', 'TMEM14A', 'EMC7', 'IDS', 'CALCOCO2', 'IRX5', 'ATXN7L3B', 'TMED7', 'NUCB1', 'CA11', 'CUL4B', 'SPG7', 'MAPK4', 'MYLIP', 'EPB41L5', 'LTBP3', 'NUMA1', 'MROH1', 'FEV', 'C5', 'LCORL', 'CMIP', 'PROCR', 'UCHL1', 'DYNLL2', 'TMED6', 'COPG1', 'ATN1', 'APOH', 'PTPA', 'OGT', 'ESYT2', 'MDM4', 'GOLPH3', 'FAM3C', 'SUMF2', 'NPNT', 'CRTAP', 'MAGED1', 'RABL6', 'STXBP1', 'EMC4', 'SCRN1', 'TMEM245', 'PPP6R2', 'UNC13A', 'NRBP2', 'LGR4', 'PEG10', 'RAB14', 'PILRB', 'GSE1', 'LASP1', 'CHD3', 'PPP3CB', 'RHBDD2', 'PTEN', 

In [51]:
class MIL(nn.Module):
    def __init__(self, all_genes):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes)
        self.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)
        self.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)
    
    def forward(self, distances, gene_expressions, current_genes):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            distance = self.distance(distance)
            gene_expression = self.gene_expression(gene_expression)
            immunogenicity, filtered_genes = self.immunogenicity(current_genes)
            alpha = self.alpha
            beta = self.beta
        
            if len(filtered_genes) == 0:
                continue  # Skip if no overlapping genes
        
        # Print debug information
            #print(f"gene_expression shape: {gene_expression.shape}")
            #print(f"current_genes length: {len(current_genes)}")
            #print(f"filtered_genes length: {len(filtered_genes)}")
        
        
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
        
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            gene_expression = gene_expression[:, gene_indices]
        
            #print(f"Filtered gene_expression shape: {gene_expression.shape}")
            #print(f"Immunogenicity shape: {immunogenicity.shape}")
        
            z = gene_expression @ immunogenicity
            #print(f"z shape: {z.shape}")
            z = z.unsqueeze(1)
            #print(f"z shape: {z.shape}")
            #print(f"distance shape: {distance}")
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            bag_output = torch.exp(alpha) * bag_output + beta
            bag_output = torch.sigmoid(bag_output)
            #print(bag_output)
            bag_outputs.append(bag_output)
            #df = pd.DataFrame(bag_outputs)
            #df.to_csv(soutput.csv')
    
        
        return torch.stack(bag_outputs).squeeze(dim=1)

In [52]:
model = MIL(all_genes)

In [53]:
output = model(distances, gene_expressions, list(gene_names[0]))
output

tensor([0.7675], grad_fn=<SqueezeBackward1>)

In [54]:

labels[0]


tensor(0.)

In [55]:
ig_scores_before_training = model.immunogenicity.ig.clone().detach().cpu()

In [56]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch

In [57]:

model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 50
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.00001)

In [58]:
current_genes = gene_names[0]

In [ ]:
for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(train_loader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, gene_names, cell_ids) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            # Convert distances and gene expressions to tensors
            distances = [d.to(device) for d in distances]
            gene_expressions = [g.to(device) for g in gene_expressions]
            label = label.clone().detach().float().to(device)
            current_genes = gene_names[0]  # Since batch_size=1

            output = model(distances, gene_expressions, current_genes)

            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())


    epoch_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    

    model.eval()
    val_loss = 0.0
    val_predictions = []
    val_labels = []
    with torch.no_grad():
        val_loss = 0.0
        val_predictions = []
        val_labels = []
        for val_distances, val_gene_expressions, val_label, val_core_idx, val_gene_names, val_cell_ids in val_loader:
            val_distances = [d.to(device) for d in val_distances]
            val_gene_expressions = [g.to(device) for g in val_gene_expressions]
            val_label = val_label.clone().detach().float().to(device)
            val_current_genes = val_gene_names[0]  # Since batch_size=1

            val_output = model(val_distances, val_gene_expressions, val_current_genes)
            val_loss += criterion(val_output, val_label).item()
            val_predictions.extend(val_output.cpu().numpy())
            val_labels.extend(val_label.cpu().numpy())

        val_loss /= len(val_loader)
        val_auroc = roc_auc_score(val_labels, val_predictions)
    
        print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_auroc:.4f}')

    
   
    early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break
   

Epoch 1/50: 100%|██████████| 7359/7359 [01:48<00:00, 67.71batch/s, loss=0.125]


Epoch [1/50], Loss: 0.4958
Validation Loss: 0.3086, Validation AUROC: 0.3732


Epoch 2/50: 100%|██████████| 7359/7359 [01:48<00:00, 67.70batch/s, loss=0.1]   


Epoch [2/50], Loss: 0.3056
Validation Loss: 0.3048, Validation AUROC: 0.4970


Epoch 3/50: 100%|██████████| 7359/7359 [01:48<00:00, 67.92batch/s, loss=0.096] 


...

Epoch 29/50: 100%|██████████| 7359/7359 [01:46<00:00, 68.78batch/s, loss=0.019]  


Epoch [29/50], Loss: 0.2086
Validation Loss: 0.2204, Validation AUROC: 0.8786


Epoch 30/50: 100%|██████████| 7359/7359 [01:47<00:00, 68.75batch/s, loss=0.0478] 


Epoch [30/50], Loss: 0.2048
Validation Loss: 0.2172, Validation AUROC: 0.8810


Epoch 31/50:   2%|▏         | 163/7359 [00:02<01:41, 70.89batch/s, loss=0.0741]

In [ ]:
import csv
ig_scores_after_training = model.immunogenicity.ig.clone().detach().cpu()
alpha = model.state_dict()['alpha'].item()
beta = model.state_dict()['beta'].item()
a = model.state_dict()['distance.a'].item()
b = model.state_dict()['gene_expression.b'].item()  
with open(os.path.join(output_dir, 'ig_score_changes.csv'), 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['alpha', alpha])
    writer.writerow(['beta', beta])
    writer.writerow(['a', a])
    writer.writerow(['b', b])
    writer.writerow(['Gene', 'IG Score Before Training', 'IG Score After Training'])
    for gene, before, after in zip(all_genes, ig_scores_before_training, ig_scores_after_training):
        writer.writerow([gene, before.item(), after.item()])
torch.save(model.state_dict(), os.path.join(output_dir, 'final_model.pth'))


In [34]:
output_dir

'pretrained_models/HumanLungCancer'

In [ ]:
model.state_dict()

OrderedDict([('alpha', tensor(0.4340)),
             ('beta', tensor(0.9409)),
             ('distance.a', tensor(-3.0000)),
             ('gene_expression.b', tensor(-0.9906)),
             ('immunogenicity.ig',
              tensor([-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]))])